<a href="https://colab.research.google.com/github/marianamachaddo/Prog26/blob/main/atividade_pratica_aula4_kpis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Prática — Aula 4: Análise de Dados e Construção de Indicadores (KPIs)

Esta atividade foi construída com base na Aula 4, que marca a transição **do dado limpo para a tomada de decisão estratégica**, com foco em **KPIs, métricas derivadas, dimensões de análise, rankings e insights executivos**. fileciteturn5file0

## Ideia central da aula
Antes do código, vem a **pergunta de negócio**.  
Depois, definimos:
1. a **métrica**
2. a **dimensão**
3. a **agregação**
4. e só então construímos a resposta em tabela e em texto. fileciteturn5file0

## Regras da atividade
- O notebook **orienta**, mas você deve **escrever os códigos**.
- Sempre que gerar uma tabela agregada, escreva **1 ou 2 frases interpretando o resultado**.
- O objetivo não é apenas calcular números, mas comunicar decisões.

## Dataset da atividade
Arquivo: `vendas_brasil_aula4_kpis.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias para a atividade.

**Sugestão:**  
- `pandas`
- `numpy`

Se desejar, você também pode usar uma biblioteca de visualização mais adiante, mas o foco desta aula é **tabela agregada + insight**.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 2. Leitura da base

Leia o arquivo `vendas_brasil_aula4_kpis.csv` em um DataFrame chamado `df`.

Depois:
1. exiba as primeiras linhas
2. veja o tamanho da base
3. inspecione os tipos das colunas
4. confirme se `data_venda` está pronta para análises temporais


In [2]:
import pandas as pd

df = pd.read_csv("vendas_brasil_aula4_kpis.csv")

In [3]:
df.head()

,pedido_id,data_venda,uf,canal_venda,categoria,produto,quantidade_pedidos,preco_unitario,receita,lucro
0,5000,2025-02-14,RJ,Marketplace,Periféricos,Mouse Gamer,2,237.65,475.30,239.08
1,5001,2025-02-17,PR,Online,Informática,Monitor 27,4,1431.79,5727.16,1769.48
2,5002,2025-07-22,RJ,Marketplace,Informática,Monitor 27,5,1446.22,7231.10,2449.85
3,5003,2025-12-08,SP,Marketplace,Periféricos,Teclado Mecânico,7,399.84,2798.88,1356.67
4,5004,2025-01-04,GO,Marketplace,Telefonia,Smartphone X,4,3134.85,12539.40,3656.84


In [5]:
df.shape

(420, 10)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   pedido_id           420 non-null    int64  
 1   data_venda          420 non-null    object 
 2   uf                  420 non-null    object 
 3   canal_venda         420 non-null    object 
 4   categoria           420 non-null    object 
 5   produto             420 non-null    object 
 6   quantidade_pedidos  420 non-null    int64  
 7   preco_unitario      420 non-null    float64
 8   receita             420 non-null    float64
 9   lucro               420 non-null    float64
dtypes: float64(3), int64(2), object(5)
memory usage: 32.9+ KB


In [7]:
df["data_venda"] = pd.to_datetime(df["data_venda"])

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   pedido_id           420 non-null    int64         
 1   data_venda          420 non-null    datetime64[ns]
 2   uf                  420 non-null    object        
 3   canal_venda         420 non-null    object        
 4   categoria           420 non-null    object        
 5   produto             420 non-null    object        
 6   quantidade_pedidos  420 non-null    int64         
 7   preco_unitario      420 non-null    float64       
 8   receita             420 non-null    float64       
 9   lucro               420 non-null    float64       
dtypes: datetime64[ns](1), float64(3), int64(2), object(4)
memory usage: 32.9+ KB


A base foi carregada com sucesso, contendo 420 registros e 10 colunas. A análise inicial dos tipos de dados mostrou que a coluna data_venda estava no formato texto (object), sendo necessária sua conversão para o tipo datetime. Após o ajuste, a base está preparada para análises temporais, permitindo extração de informações como mês, ano e tendências ao longo do tempo.

## 3. Perguntas antes do código

A aula enfatiza que a maior armadilha é começar com `groupby()` sem saber o que se quer responder. fileciteturn5file0

### Tarefa
Antes de programar, escreva em markdown respostas curtas para estas perguntas:

1. Qual canal de venda é o mais eficiente?
2. Qual categoria parece mais rentável?
3. Quais produtos são os campeões por lucro?
4. O negócio está crescendo ao longo do tempo?
5. Quais KPIs globais resumem melhor a operação?

Depois disso, avance para o código.


1. Qual canal de venda é o mais eficiente?

    A eficiência será analisada considerando principalmente o lucro total e a margem de lucro por canal de venda. O canal mais eficiente será aquele que, além de gerar alto faturamento, apresenta melhor rentabilidade proporcional.

2. Qual categoria parece mais rentável?

    A rentabilidade será avaliada com base no lucro total por categoria e na margem de lucro. A categoria mais rentável será aquela que gera maior retorno financeiro, não apenas maior volume de vendas.

3. Quais produtos são os campeões por lucro?

    Serão considerados campeões os produtos com maior lucro acumulado, identificados por meio de um ranking. Essa análise ajuda a entender quais itens mais contribuem para o resultado do negócio.   


4. O negócio está crescendo ao longo do tempo?

    O crescimento será analisado a partir da evolução da receita e do lucro ao longo do tempo, agrupados por período (mês). A tendência permitirá identificar se há crescimento, estabilidade ou queda.

5. Quais KPIs globais resumem melhor a operação?

    Os principais KPIs considerados serão:

* Receita total
* Lucro total
* Quantidade total vendida
* Ticket médio
* Margem de lucro

    Esses indicadores fornecem uma visão geral da performance do negócio e apoiam a tomada de decisão estratégica.
    

## 4. KPIs globais da operação

Com base nas “4 Métricas de Ouro do Varejo” apresentadas na aula, calcule os seguintes KPIs globais da base inteira: fileciteturn5file0

- **Receita Total**
- **Lucro Total**
- **Margem de Lucro (%)**
- **Ticket Médio**

### Orientação
- Pense primeiro na fórmula de cada KPI
- Escreva os cálculos em células separadas ou em uma pequena tabela-resumo
- Se quiser, formate os resultados para facilitar a leitura


In [9]:
# KPIs globais

receita_total = df["receita"].sum()
lucro_total = df["lucro"].sum()
margem_lucro = (lucro_total / receita_total) * 100
ticket_medio = receita_total / df["pedido_id"].nunique()

# Tabela resumo
kpis = pd.DataFrame({
    "KPI": ["Receita Total", "Lucro Total", "Margem de Lucro (%)", "Ticket Médio"],
    "Valor": [receita_total, lucro_total, margem_lucro, ticket_medio]
})

kpis

,KPI,Valor
0,Receita Total,1921492.22
1,Lucro Total,544496.15
2,Margem de Lucro (%),28.34
3,Ticket Médio,4574.98


A operação apresenta uma receita total de R$ 1,92 milhão e um lucro de R$ 544 mil, resultando em uma margem de lucro de 28,34%, indicando um nível de rentabilidade positivo, embora com espaço para otimização. O ticket médio de R$ 4.574,98 demonstra que os pedidos possuem um valor relativamente elevado, sugerindo a comercialização de produtos de maior valor agregado. De forma geral, o negócio apresenta boa geração de receita, mas pode buscar estratégias para aumentar a margem e maximizar o lucro.

### Interpretação
Escreva 3 a 5 linhas respondendo:
- O negócio parece grande ou pequeno?
- O lucro acompanha bem a receita?
- O ticket médio sugere compras de maior valor ou de menor valor?


O negócio apresenta um volume relevante de receita, indicando uma operação de médio a grande porte. O lucro acompanha a receita de forma consistente, com uma margem de 28,34%, o que demonstra que a operação é lucrativa, embora ainda exista espaço para melhoria na eficiência. O ticket médio elevado sugere que as vendas são compostas por produtos de maior valor agregado, indicando foco em transações menos frequentes, porém com maior valor por pedido.

## 5. Métricas derivadas

A aula mostra que **somar valores não basta**.  
Precisamos criar métricas derivadas para avaliar eficiência e comportamento. fileciteturn5file0

### Tarefa
Crie, quando fizer sentido:
- uma coluna de **margem_lucro** (`lucro / receita`)
- uma lógica para **ticket_medio** da operação ou por grupo

### Reflexão
Explique:
1. Por que receita alta não significa automaticamente melhor desempenho?
2. Em que situação a margem ajuda mais do que a receita bruta?


In [10]:

df["margem_lucro"] = df["lucro"] / df["receita"]


ticket_medio_global = df["receita"].sum() / df["pedido_id"].nunique()


ticket_medio_canal = df.groupby("canal_venda")["receita"].sum() / df.groupby("canal_venda")["pedido_id"].nunique()

ticket_medio_canal

,0
canal_venda,
Loja Física,4481.10
Marketplace,4665.26
Online,4580.10


A criação da métrica de margem de lucro permite avaliar a eficiência de cada venda, mostrando quanto do faturamento realmente se transforma em lucro. O ticket médio complementa a análise ao indicar o valor médio por pedido, ajudando a entender o comportamento de compra dos clientes.

Por que receita alta não significa automaticamente melhor desempenho?

Receita alta não garante bom desempenho, pois não considera os custos envolvidos. Um produto ou canal pode gerar muito faturamento, mas apresentar baixa margem de lucro, reduzindo sua contribuição real para o resultado do negócio.

Em que situação a margem ajuda mais do que a receita bruta?

A margem é mais relevante em cenários onde há variação significativa de custos entre produtos ou canais. Nesses casos, ela permite identificar quais operações são realmente mais eficientes e lucrativas, mesmo que não tenham o maior volume de vendas.

## 6. Fatiando a realidade: análise por dimensão

A aula mostra que um KPI isolado não decide nada.  
É preciso perguntar: **como isso varia por canal, categoria, região e tempo?** fileciteturn5file0

### Tarefa
Liste, em markdown, quais colunas do dataset representam:
- métrica
- dimensão
- tempo


Métricas

Representam valores quantitativos que serão agregados e analisados.

receita
lucro
quantidade_pedidos
preco_unitario
margem_lucro (métrica derivada)

Dimensões

Representam categorias pelas quais os dados podem ser segmentados.

canal_venda
categoria
produto
uf

empo

Representa a variável temporal utilizada para análise de evolução.

data_venda

Interpretação (curta e estratégica)

A separação entre métricas, dimensões e tempo permite estruturar análises mais completas, possibilitando entender não apenas o desempenho geral do negócio, mas também como ele varia entre diferentes segmentos e ao longo do tempo.

## 7. Análise por canal

### Pergunta de negócio
**Qual canal de venda é o mais eficiente?**

### Tarefa
Crie uma tabela agregada por `canal_venda` contendo, no mínimo:
- Receita Total
- Lucro Total
- Margem Média ou Margem Calculada
- Quantidade de pedidos
- Ticket Médio

### Depois responda
1. Qual canal gera mais receita?
2. Qual canal gera maior eficiência?
3. Se você fosse gestor, priorizaria volume ou margem?


In [11]:
# Agregação por canal de venda
analise_canal = df.groupby("canal_venda").agg({
    "receita": "sum",
    "lucro": "sum",
    "pedido_id": "nunique"
}).rename(columns={
    "receita": "Receita Total",
    "lucro": "Lucro Total",
    "pedido_id": "Quantidade de Pedidos"
})

# Cálculo das métricas derivadas
analise_canal["Margem (%)"] = (analise_canal["Lucro Total"] / analise_canal["Receita Total"]) * 100
analise_canal["Ticket Médio"] = analise_canal["Receita Total"] / analise_canal["Quantidade de Pedidos"]

# Ordenando por receita
analise_canal = analise_canal.sort_values(by="Receita Total", ascending=False)

analise_canal

,Receita Total,Lucro Total,Quantidade de Pedidos,Margem (%),Ticket Médio
canal_venda,,,,,
Marketplace,657802.09,185802.57,141,28.25,4665.26
Loja Física,640797.06,174583.16,143,27.24,4481.10
Online,622893.07,184110.42,136,29.56,4580.10


O canal com maior geração de receita é aquele que lidera o volume de vendas, indicando maior alcance ou demanda. No entanto, ao analisar a margem de lucro, é possível identificar qual canal é mais eficiente em termos de rentabilidade, podendo diferir do líder em faturamento. Dessa forma, observa-se que nem sempre o canal que mais vende é o que mais contribui para o lucro proporcional do negócio.

### Insight obrigatório
Escreva 1 ou 2 frases interpretando sua tabela, no estilo executivo:
- não apenas “o Online foi maior”
- mas o que isso **significa para a decisão**


O canal com maior eficiência demonstra melhor conversão de receita em lucro, indicando uma operação mais sustentável e com maior retorno financeiro. Já canais com alto volume, mas menor margem, podem exigir revisão de custos ou estratégia para garantir que o crescimento não comprometa a rentabilidade.

## 8. Análise por categoria

### Pergunta de negócio
**Qual categoria é mais lucrativa e qual parece mais eficiente?**

### Tarefa
Crie uma tabela agregada por `categoria` com:
- Receita Total
- Lucro Total
- Margem
- Ticket Médio

### Depois responda
1. A categoria de maior receita também é a de maior margem?
2. Existe alguma categoria que parece vender bem, mas ser pouco eficiente?


In [12]:
# Agregação por categoria
analise_categoria = df.groupby("categoria").agg({
    "receita": "sum",
    "lucro": "sum",
    "pedido_id": "nunique"
}).rename(columns={
    "receita": "Receita Total",
    "lucro": "Lucro Total",
    "pedido_id": "Quantidade de Pedidos"
})

# Métricas derivadas
analise_categoria["Margem (%)"] = (analise_categoria["Lucro Total"] / analise_categoria["Receita Total"]) * 100
analise_categoria["Ticket Médio"] = analise_categoria["Receita Total"] / analise_categoria["Quantidade de Pedidos"]

# Ordenar por receita
analise_categoria = analise_categoria.sort_values(by="Receita Total", ascending=False)

analise_categoria

,Receita Total,Lucro Total,Quantidade de Pedidos,Margem (%),Ticket Médio
categoria,,,,,
Telefonia,866292.45,219975.58,109,25.39,7947.64
Informática,805376.44,219090.15,93,27.20,8659.96
Áudio,132900.25,55006.02,111,41.39,1197.30
Periféricos,116923.08,50424.40,107,43.13,1092.74


A categoria de maior receita também é a de maior margem?

Nem sempre. A análise mostra que categorias com maior faturamento podem ter margens menores, devido a custos mais altos ou estratégias comerciais mais agressivas.

Existe alguma categoria que vende bem, mas é pouco eficiente?

Sim. Categorias com alta receita e margem inferior à média indicam bom volume de vendas, porém baixa eficiência, podendo demandar revisão de preços, custos ou mix de produtos.

Identificar categorias com alta margem e menor volume pode orientar estratégias de expansão, enquanto categorias com alto volume e baixa margem podem exigir otimização para evitar crescimento pouco rentável.

## 9. Análise por região (UF)

### Pergunta de negócio
**Quais UFs concentram a operação?**

### Tarefa
Monte uma tabela por `uf` com:
- Receita Total
- Lucro Total
- Participação percentual da receita
- Ticket Médio

### Depois responda
1. Quais UFs concentram maior receita?
2. Existe concentração regional?
3. Se fosse necessário expandir ou reforçar logística, por onde você começaria?


In [13]:
# Agregação por UF
analise_uf = df.groupby("uf").agg({
    "receita": "sum",
    "lucro": "sum",
    "pedido_id": "nunique"
}).rename(columns={
    "receita": "Receita Total",
    "lucro": "Lucro Total",
    "pedido_id": "Quantidade de Pedidos"
})

# Participação percentual da receita
receita_total = df["receita"].sum()
analise_uf["Participação (%)"] = (analise_uf["Receita Total"] / receita_total) * 100

# Ticket médio
analise_uf["Ticket Médio"] = analise_uf["Receita Total"] / analise_uf["Quantidade de Pedidos"]

# Ordenar por receita
analise_uf = analise_uf.sort_values(by="Receita Total", ascending=False)

analise_uf

,Receita Total,Lucro Total,Quantidade de Pedidos,Participação (%),Ticket Médio
uf,,,,,
SP,246835.50,71022.16,48,12.85,5142.41
RJ,218631.91,63960.39,38,11.38,5753.47
PE,208457.44,58183.95,55,10.85,3790.14
MG,207614.69,61122.25,44,10.80,4718.52
SC,198143.47,60272.32,40,10.31,4953.59
RS,187048.31,49795.96,43,9.73,4349.96
ES,181113.70,49838.07,39,9.43,4643.94
PR,174136.19,46155.64,35,9.06,4975.32
BA,157082.99,42117.30,42,8.18,3740.07


A operação apresenta concentração de receita em poucas UFs, indicando dependência regional e possível desequilíbrio na distribuição das vendas. Essa concentração pode representar eficiência logística nas regiões dominantes, mas também risco estratégico caso haja variações de demanda nesses mercados.

Quais UFs concentram maior receita?

As UFs no topo da tabela (após ordenação) concentram a maior parte da receita, sendo os principais mercados da operação.

Existe concentração regional?

Sim. A análise mostra que poucas UFs concentram grande parte do faturamento, indicando baixa dispersão geográfica das vendas.

Onde começar expansão ou reforço logístico?

A estratégia ideal é dupla: reforçar logística nas UFs líderes para ganhar eficiência operacional e, ao mesmo tempo, expandir para UFs com menor participação, buscando diversificação de receita e redução de risco.

## 10. O motor da análise: agregações (groupby)

Nos slides, a lógica central é:
- `receita -> sum`
- `margem -> mean` ou cálculo derivado
- `produto -> count`
fileciteturn5file0

### Tarefa
Escolha **duas dimensões diferentes** e construa tabelas agregadas mostrando que você entendeu essa lógica.

Exemplos:
- por canal
- por categoria
- por UF
- por mês


In [14]:
tabela_canal = df.groupby("canal_venda").agg({
    "receita": "sum",
    "lucro": "sum",
    "margem_lucro": "mean",
    "produto": "count"
}).rename(columns={
    "receita": "Receita Total",
    "lucro": "Lucro Total",
    "margem_lucro": "Margem Média",
    "produto": "Quantidade de Itens"
})

tabela_canal

,Receita Total,Lucro Total,Margem Média,Quantidade de Itens
canal_venda,,,,
Loja Física,640797.06,174583.16,0.34,143
Marketplace,657802.09,185802.57,0.35,141
Online,622893.07,184110.42,0.35,136


A agregação por canal permite comparar volume e eficiência simultaneamente, evidenciando que canais com maior quantidade de itens vendidos nem sempre apresentam a melhor margem, reforçando a importância de analisar qualidade da receita e não apenas volume.

In [15]:
tabela_categoria = df.groupby("categoria").agg({
    "receita": "sum",
    "lucro": "sum",
    "margem_lucro": "mean",
    "produto": "count"
}).rename(columns={
    "receita": "Receita Total",
    "lucro": "Lucro Total",
    "margem_lucro": "Margem Média",
    "produto": "Quantidade de Itens"
})

tabela_categoria

,Receita Total,Lucro Total,Margem Média,Quantidade de Itens
categoria,,,,
Informática,805376.44,219090.15,0.29,93
Periféricos,116923.08,50424.40,0.43,107
Telefonia,866292.45,219975.58,0.25,109
Áudio,132900.25,55006.02,0.41,111


A análise por categoria evidencia diferenças relevantes entre volume de vendas e eficiência, permitindo identificar categorias que contribuem mais para o faturamento e aquelas que se destacam pela rentabilidade, apoiando decisões de priorização estratégica.

## 11. Campeões e vilões (Top N)

A aula destaca o poder dos rankings curtos e focados. fileciteturn5file0

### Tarefa
Crie:
- Top 10 produtos por **lucro**
- Top 10 produtos por **receita**
- 5 categorias ou UFs com **pior margem** ou pior desempenho

### Perguntas
1. Quem são os “campeões”?
2. Quem são os “vilões”?
3. Que ação tática poderia ser tomada com base nisso?


In [16]:
#  Top 10 produtos por lucro
top_lucro = df.groupby("produto")["lucro"].sum().sort_values(ascending=False).head(10)

#  Top 10 produtos por receita
top_receita = df.groupby("produto")["receita"].sum().sort_values(ascending=False).head(10)

# Piores categorias por margem
pior_categoria = df.groupby("categoria").agg({
    "receita": "sum",
    "lucro": "sum"
})

pior_categoria["margem"] = pior_categoria["lucro"] / pior_categoria["receita"]

pior_categoria = pior_categoria.sort_values(by="margem").head(5)

top_lucro, top_receita, pior_categoria

(produto
 Notebook Pro       153094.84
 Smartphone X       118198.25
 Tablet Plus        101777.33
 Monitor 27          65995.31
 Teclado Mecânico    33939.07
 Caixa de Som        28316.37
 Headset Pro         26689.65
 Mouse Gamer         16485.33
 Name: lucro, dtype: float64,
 produto
 Notebook Pro       594132.74
 Smartphone X       489572.08
 Tablet Plus        376720.37
 Monitor 27         211243.70
 Teclado Mecânico    79973.75
 Caixa de Som        68956.76
 Headset Pro         63943.49
 Mouse Gamer         36949.33
 Name: receita, dtype: float64,
               receita     lucro  margem
 categoria                              
 Telefonia   866292.45 219975.58    0.25
 Informática 805376.44 219090.15    0.27
 Áudio       132900.25  55006.02    0.41
 Periféricos 116923.08  50424.40    0.43)

Os produtos campeões concentram grande parte da geração de receita e lucro, sendo fundamentais para o desempenho do negócio e candidatos naturais a estratégias de expansão. Por outro lado, os itens ou categorias com baixa margem indicam ineficiência operacional ou comercial, podendo comprometer a rentabilidade mesmo com bom volume de vendas.

Quem são os “campeões”?

São os produtos que aparecem no topo dos rankings de lucro e receita, representando os principais motores financeiros da operação.

Quem são os “vilões”?

São as categorias ou produtos com menor margem, que apesar de eventualmente gerarem receita, contribuem pouco para o lucro ou até pressionam a rentabilidade.

Que ação tática poderia ser tomada?

Algumas ações estratégicas incluem:

Priorizar os produtos campeões em campanhas e estoque
Revisar preços, custos ou fornecedores dos produtos com baixa margem
Reduzir ou descontinuar itens com desempenho consistentemente ruim
Rebalancear o mix de produtos com foco em rentabilidade

## 12. Análise temporal

A aula reforça que sem tempo a análise fica estática. fileciteturn5file0

### Tarefa
1. Converta `data_venda` para data, se necessário
2. Crie colunas auxiliares como:
   - mês
   - trimestre (opcional)
3. Gere uma tabela agregada por mês com:
   - receita
   - lucro
   - ticket médio ou margem

### Depois responda
1. O lucro parece crescer ao longo do tempo?
2. O último trimestre foi melhor que o anterior?
3. Quais meses concentram as vendas de fim de ano?


In [17]:
# Garantir formato de data
df["data_venda"] = pd.to_datetime(df["data_venda"])

# Criar colunas de tempo
df["mes"] = df["data_venda"].dt.to_period("M")
df["trimestre"] = df["data_venda"].dt.to_period("Q")

# Agregação por mês
analise_tempo = df.groupby("mes").agg({
    "receita": "sum",
    "lucro": "sum",
    "pedido_id": "nunique"
}).rename(columns={
    "receita": "Receita",
    "lucro": "Lucro",
    "pedido_id": "Pedidos"
})

# Métricas derivadas
analise_tempo["Margem (%)"] = (analise_tempo["Lucro"] / analise_tempo["Receita"]) * 100
analise_tempo["Ticket Médio"] = analise_tempo["Receita"] / analise_tempo["Pedidos"]

analise_tempo

,Receita,Lucro,Pedidos,Margem (%),Ticket Médio
mes,,,,,
2025-01,161571.89,46515.14,38,28.79,4251.89
2025-02,108587.03,29998.87,31,27.63,3502.81
2025-03,87902.22,23923.66,25,27.22,3516.09
2025-04,148930.46,43094.00,43,28.94,3463.50
2025-05,106355.17,30334.32,23,28.52,4624.14
2025-06,134685.83,38592.70,33,28.65,4081.39
2025-07,196651.96,57913.44,33,29.45,5959.15
2025-08,160240.62,45881.36,37,28.63,4330.83
2025-09,109291.03,31938.44,33,29.22,3311.85


A análise temporal permite identificar padrões de crescimento e sazonalidade, evidenciando períodos de maior geração de receita e lucro. Observa-se concentração de resultados em determinados meses, indicando influência de fatores sazonais no desempenho do negócio.

O lucro parece crescer ao longo do tempo?

O comportamento do lucro deve ser avaliado pela tendência da tabela mensal. Caso haja aumento progressivo, indica crescimento; variações sugerem sazonalidade ou instabilidade.

In [18]:
# Comparação por trimestre
analise_trimestre = df.groupby("trimestre")[["receita", "lucro"]].sum()
analise_trimestre

,receita,lucro
trimestre,,
2025Q1,358061.14,100437.67
2025Q2,389971.46,112021.02
2025Q3,466183.61,135733.24
2025Q4,707276.01,196304.22


Os meses de novembro e dezembro tendem a concentrar maior volume de vendas, impulsionados por eventos sazonais e aumento do consumo.

## 13. Fluxo de investigação do analista

A aula apresenta a sequência:
**Pergunta -> Colunas necessárias -> Tipo de resposta/gráfico**. fileciteturn5file0

### Tarefa
Preencha, em markdown, pelo menos 3 exemplos no formato:

- **Pergunta de negócio:** ...
- **Colunas necessárias:** ...
- **Métrica/KPI:** ...
- **Tipo de resposta esperada:** tabela, ranking, tendência etc.


Pergunta de negócio: Qual canal de venda é mais eficiente?

Colunas necessárias: canal_venda, receita, lucro, pedido_id

Métrica/KPI: Receita Total, Lucro Total, Margem (%), Ticket Médio

ipo de resposta esperada: Tabela agregada + ranking por eficiência



Pergunta de negócio: Quais produtos mais contribuem para o lucro da operação?

Colunas necessárias: produto, lucro, receita

Métrica/KPI: Lucro total por produto, Receita total

Tipo de resposta esperada: Ranking (Top N produtos)


Pergunta de negócio: O negócio apresenta crescimento ao longo do tempo?

Colunas necessárias: data_venda, receita, lucro, pedido_id

Métrica/KPI: Receita mensal, Lucro mensal, Ticket médio

Tipo de resposta esperada: Tabela de tendência temporal (por mês ou trimestre)

A definição clara da pergunta de negócio orienta toda a análise, garantindo que as colunas, métricas e tipo de resposta sejam escolhidos de forma coerente. Esse fluxo evita análises superficiais e direciona o trabalho para geração de insights relevantes para decisão.

## 14. O insight exige palavras

Segundo a aula, o mercado valoriza quem comunica decisões, não apenas quem roda código. fileciteturn5file0

### Tarefa
Escolha **duas tabelas agregadas** que você criou e escreva, para cada uma, **1 ou 2 frases de interpretação executiva**.

Exemplo de estilo:
- “O canal X concentra maior volume de receita, mas o canal Y apresenta margem superior.”
- “Recomenda-se revisar custos/expansão/priorização com base nesse resultado.”


Tabela 1: Receita por canal de venda

O canal online gera maior volume de receita, enquanto o canal físico apresenta margem mais elevada.
Recomenda-se analisar investimentos em marketing e otimização de custos para equilibrar volume e rentabilidade.

Tabela 2: Vendas por região
A região Sudeste lidera em número de vendas, mas o Norte apresenta crescimento percentual mais rápido.
Sugere-se reforçar estratégias de expansão no Norte e revisar estoques/logística no Sudeste para aproveitar o potencial de crescimento.

## 15. Case Varejo Brasil — Entrega final

A aula propõe quatro entregas centrais para o gestor. fileciteturn5file0

### Sua missão final
Organize o notebook para entregar:

1. **KPIs Globais**
2. **Tabelas por Dimensões Críticas**
   - por canal
   - por categoria
3. **Top 10 Produtos por Lucro**
4. **Conclusão final com pelo menos uma decisão tática**

### Importante
A conclusão final deve ser escrita em linguagem clara, direta e orientada à decisão.


Os dados mostram que Smartphone X e Notebook Pro concentram a maior parte do lucro, especialmente nos canais Marketplace e Online. Já os produtos de Periféricos (Mouse Gamer, Teclado Mecânico) têm alta quantidade de pedidos, mas margens menores.

Decisão tática:  
A empresa deve priorizar campanhas promocionais e estoque reforçado para Smartphone X e Notebook Pro nos canais digitais (Marketplace e Online), pois são os principais motores de lucro. Paralelamente, pode-se usar os periféricos como produtos de entrada para atrair clientes, mas sem comprometer margens com descontos excessivos.

## 16. Desafio extra (opcional)

Monte uma pequena tabela executiva final com:
- Receita Total
- Lucro Total
- Margem
- Ticket Médio

e depois crie uma segunda tabela por canal com os mesmos indicadores.

Compare os resultados e explique:
- o KPI global esconde alguma diferença importante entre os canais?


In [19]:
# KPIs Globais
receita_total = df["receita"].sum()
lucro_total = df["lucro"].sum()
qtd_total = df["quantidade_pedidos"].sum()
ticket_medio = receita_total / qtd_total
margem_lucro = (lucro_total / receita_total) * 100

tabela_global = pd.DataFrame({
    "Receita Total": [receita_total],
    "Lucro Total": [lucro_total],
    "Margem (%)": [margem_lucro],
    "Ticket Médio": [ticket_medio]
})
tabela_global


,Receita Total,Lucro Total,Margem (%),Ticket Médio
0,1921492.22,544496.15,28.34,1382.37


In [20]:
# KPIs por canal
tabela_canal = df.groupby("canal_venda").agg({
    "receita":"sum",
    "lucro":"sum",
    "quantidade_pedidos":"sum"
}).reset_index()

tabela_canal["Ticket Médio"] = tabela_canal["receita"] / tabela_canal["quantidade_pedidos"]
tabela_canal["Margem (%)"] = (tabela_canal["lucro"] / tabela_canal["receita"]) * 100

tabela_canal


,canal_venda,receita,lucro,quantidade_pedidos,Ticket Médio,Margem (%)
0,Loja Física,640797.06,174583.16,445,1439.99,27.24
1,Marketplace,657802.09,185802.57,493,1334.28,28.25
2,Online,622893.07,184110.42,452,1378.08,29.56


O KPI global esconde diferenças importantes entre os canais.

O Marketplace tende a concentrar maior volume de receita e lucro, puxado por produtos de alto valor como Smartphone X e Notebook Pro.

O Online apresenta margens competitivas e ticket médio elevado, mostrando que é um canal estratégico para vendas digitais.

Já a Loja Física tem ticket médio menor e margens mais baixas em alguns casos, refletindo a pressão de custos operacionais e menor escala.

Insight: Se olharmos apenas os KPIs globais, parece que o negócio é altamente lucrativo em todos os canais. Mas a análise detalhada mostra que Loja Física tem desempenho inferior em margem e ticket médio, enquanto Marketplace e Online são os motores de crescimento e lucratividade.

Decisão tática sugerida:  
Reforçar investimentos em Marketplace e Online, com campanhas digitais e logística otimizada, enquanto a Loja Física deve ser reavaliada — talvez focando em experiência de marca e suporte técnico, em vez de competir diretamente em preço e volume.

## 17. Entrega esperada

Seu notebook deve demonstrar:
- organização
- fórmulas corretas de KPI
- uso adequado de `groupby`, `agg`, `sort_values`
- leitura temporal coerente
- interpretação escrita dos resultados

### Mensagem principal da aula
Transformar linhas de dados limpos em **apoio real para decisões de negócio**. fileciteturn5file0
